In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, display
pd.set_option("display.width", 160)
OUT = ROOT / "outputs"
def show(name): display(Image(filename=str(OUT / "figures" / f"{name}.png")))


# 03 Volatility target and features
Target at *t* = annualised RMS of returns *t+1..t+5*. Features at *t* use data up to *t* only.

In [2]:
from src.data.align import load_processed
from src.features.build_features import build_dataset
from src.features.returns import equal_weight_return
ind, aux = load_processed()
r = equal_weight_return(ind)
data = build_dataset(r, aux["vix_close"], horizon=5)
data.dropna().tail()

,ret_lag0,ret_lag1,ret_lag2,ret_lag3,ret_lag4,cumret_5,vol_5,cumret_21,vol_21,cumret_63,vol_63,abs_ret,vix_level,vix_chg_1,vix_chg_5,drawdown_252,vix_vol_ratio,target
date,,,,,,,,,,,,,,,,,,
2026-08-18,0.00127,-0.00513,0.00260,0.00674,0.00006,0.005515,0.063546,0.032207,0.104922,0.039816,0.094803,0.00127,15.84,0.041901,0.035994,-0.003867,0.411911,0.105218
2026-08-19,0.00904,0.00127,-0.00513,0.00260,0.00674,0.014544,0.090314,0.034432,0.106878,0.051108,0.096445,0.00904,14.89,-0.061849,0.023099,0.000000,0.331591,0.084406
2026-08-20,-0.00942,0.00904,0.00127,-0.00513,0.00260,-0.001741,0.101683,0.022275,0.111449,0.034000,0.097274,0.00942,16.01,0.072524,0.090139,-0.009420,0.362234,0.056607
2026-08-21,0.00683,-0.00942,0.00904,0.00127,-0.00513,0.002471,0.111130,0.048679,0.094153,0.039669,0.098192,0.00683,15.13,-0.056534,0.059923,-0.002654,0.474347,0.029460
2026-08-24,-0.00148,0.00683,-0.00942,0.00904,0.00127,0.006148,0.105517,0.043256,0.093412,0.031631,0.097425,0.00148,15.85,0.046490,0.042532,-0.004130,0.528733,0.027725


In [3]:
# leakage check: wreck the future after t and confirm features at <= t are unchanged
t = 4000
r2, v2 = r.copy(), aux["vix_close"].copy()
r2.iloc[t + 1:] *= 5; v2.iloc[t + 1:] *= 3
alt = build_dataset(r2, v2, horizon=5)
feats = [c for c in data.columns if c != "target"]
print("features identical up to t:", data[feats].iloc[:t + 1].equals(alt[feats].iloc[:t + 1]))
print("target at t unchanged by returns <= t:", np.isclose(data.target.iloc[t], build_dataset(r.where(r.index > r.index[t], 0.123), aux["vix_close"]).target.iloc[t]))

features identical up to t: True
target at t unchanged by returns <= t: True


In [4]:
dd = pd.DataFrame({"feature": feats})
dd["description"] = ["see docs/data_dictionary.md"] * len(feats)
print(len(feats), "features:", ", ".join(feats))

17 features: ret_lag0, ret_lag1, ret_lag2, ret_lag3, ret_lag4, cumret_5, vol_5, cumret_21, vol_21, cumret_63, vol_63, abs_ret, vix_level, vix_chg_1, vix_chg_5, drawdown_252, vix_vol_ratio


In [5]:
data[['target']].dropna().describe().T

,count,mean,std,min,25%,50%,75%,max
target,9228.0,0.137059,0.099225,0.011715,0.07892,0.112745,0.16524,1.351581
